# 81일차 · 실행 추적과 기준선 설정

7월 1일 문의에 지난달 기한인 30일을 안내했습니다. 검색기가 구판을 골랐는지, 모델이 개정판을 읽고도 잘못 답했는지 실행을 나눠 살펴봅니다.

LangSmith에서 검색·모델·도구 실행을 따라가고 실패가 처음 발생한 단계를 찾는다. 데이터셋·모델·프롬프트·색인 버전과 기준선 실험을 기록한다.

## 어느 단계에서 구판이 들어왔나요?

In [ ]:
editions = [("A1", "2026-06-01", 30), ("A2", "2026-07-01", 90)]
def retrieve(as_of):
    eligible = [row for row in editions if row[1] <= as_of]
    return eligible[0] if eligible else None

def answer(as_of):
    doc = retrieve(as_of)
    return {"citation": doc[0], "days": doc[2]} if doc else {"citation": None}

print(retrieve("2026-07-01"))
print(answer("2026-07-01"))

('A1', '2026-06-01', 30)
{'citation': 'A1', 'days': 30}


두 번째 문서도 7월 1일부터 유효합니다. 첫 번째 후보를 그대로 돌려주면서 A1이 선택됐고 답변은 그 값을 그대로 썼습니다. 검색 결과에서 이미 오류를 볼 수 있습니다.

### 직접 해 보기

기준일에 유효한 문서 중 가장 최근 문서를 고르세요. 6월 30일, 7월 1일, 최초 시행 전날에 각각 무엇을 반환하는지 확인하세요.

In [ ]:
# retrieve를 다시 정의하고 세 날짜를 검사하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>def retrieve(as_of):
    eligible = [row for row in editions if row[1] &lt;= as_of]
    return max(eligible, key=lambda row: row[1], default=None)

for day in [&quot;2026-06-30&quot;, &quot;2026-07-01&quot;, &quot;2026-05-31&quot;]:
    print(day, answer(day))</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>2026-06-30 {&#x27;citation&#x27;: &#x27;A1&#x27;, &#x27;days&#x27;: 30}
2026-07-01 {&#x27;citation&#x27;: &#x27;A2&#x27;, &#x27;days&#x27;: 90}
2026-05-31 {&#x27;citation&#x27;: None}
</code></pre></details>

## 수업 서비스에 적용하기

수업용 LangSmith 프로젝트와 인증 정보를 설정하고 langsmith 패키지를 준비하세요. 아래 셀은 현재 커널의 두 함수를 한 요청의 부모·자식 실행으로 기록합니다. 추적을 켠 뒤 기준선과 수정본을 각각 실행해 입력, 검색 결과, 최종 응답을 비교하세요. 팀의 그래프에서는 검색·모델·도구 노드를 열고 데이터셋·모델·프롬프트·색인 버전과 실행 링크를 함께 남기세요.

In [ ]:
run_integration = False
if run_integration:
    from langsmith import traceable, tracing_context
    
    traced_search = traceable(name="상품설명서 검색", run_type="retriever")(retrieve)
    @traceable(name="문의 처리")
    def traced_answer(as_of):
        doc = traced_search(as_of)
        return {"citation": doc[0], "days": doc[2]} if doc else {"citation": None}
    
    with tracing_context(enabled=True, project_name="course-product-qa"):
        print(traced_answer("2026-07-01"))

### 제출 전 확인

LangSmith에서 검색·모델·도구 실행을 따라가고 실패가 처음 발생한 단계를 찾는다. 데이터셋·모델·프롬프트·색인 버전과 기준선 실험을 기록한다.

[공식 문서](https://docs.langchain.com/langsmith/annotate-code)